In [ ]:
import kagglehub

from pathlib import Path
import pandas as pd
from tokenizers import BertWordPieceTokenizer

In [ ]:

data_path = Path("../data")
if not data_path.exists():
    data_path.mkdir(parents=True)

ckpt_path = Path("../ckpt")
if not ckpt_path.exists():
    ckpt_path.mkdir(parents=True)

tokenizer_path = ckpt_path/"ch3_tokenizer"
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [ ]:

dataset_path = Path(
    kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
)

In [ ]:
corpus_path = data_path / "corpus.txt"
imdb_df = pd.read_csv(dataset_path / "IMDB Dataset.csv")


reviews = imdb_df.review.to_string(index=None) 
with open(corpus_path, "w") as f: 
    f.writelines(reviews) 

In [ ]:
bert_wordpiece_tokenizer = BertWordPieceTokenizer() 
bert_wordpiece_tokenizer.train(str(corpus_path)) 

In [ ]:
bert_wordpiece_tokenizer.get_vocab()

In [ ]:


bert_wordpiece_tokenizer.save_model(str(tokenizer_path))

bert_wordpiece_tokenizer = BertWordPieceTokenizer.from_file(str(tokenizer_path / "vocab.txt"))

In [ ]:

bert_wordpiece_tokenizer.encode("it works or not ohoh").tokens

In [ ]:
bert_wordpiece_tokenizer.encode("it works or not").tokens


In [ ]:
# from transformers

from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained(str(tokenizer_path / "vocab.txt"), padding_side="right")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("text", data_files=str(corpus_path))

In [ ]:
dataset["train"]["text"][0]

In [ ]:
tokenized_dataset = dataset.map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=128),
)

In [ ]:
tokenized_dataset["train"][2000]

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
data_collator

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="BERT",
    num_train_epochs=1,
    per_device_train_batch_size=128,
)

In [ ]:
from transformers import BertConfig, BertForMaskedLM

bert = BertForMaskedLM(BertConfig())

In [ ]:
tokenized_dataset["train"]["text"]


In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=bert,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"]
)

In [ ]:
trainer.train()